# Parse messages\nSplit text records into classified, protocol-neutral message rows.

In [ ]:
project_root = "."
source = "data/capture"
fix_dictionary = "data/fix"
protocols = None
pattern = "*.log*"
header = None
recursive = True
spill = False
timezone = None
include_regexes = []
exclude_regexes = []
include_msgtypes = []
exclude_msgtypes = []
technical_plugins = ["jolokia"]
start = None
end = None
duration_ns = None
catalog = "rekep"
catalog_properties = {}
table_properties = {"history.expire.max-snapshot-age-ms": "604800000"}
branch = "root"
target = "logs.messages"
merge_by = True
batch_row_size = 65_536
batch_byte_size = 67_108_864
max_row_byte_size = 67_108_864
commit_row_size = 250_000
limit = None
log_level = "INFO"

In [ ]:
import pyarrow
import pyarrow.compute as pc
import pyarrow.fs
from rekep.filesystems import resolve
from rekep.fix import FixRegistry
from rekep.fix.rules import Rules
from rekep.iceberg import IcebergDataset
from rekep.text import TextFile, TextFiles
from rekep.times import unix_of
from rekep.urls import Url
from rekep.logs import Stage, configure

configure(log_level)
lower, upper = unix_of(start), unix_of(end, upper=True)
registry = FixRegistry(
    cache_dir=Url.from_string(str(fix_dictionary)).resolve(project_root),
    announce=print,
)


declared = {
    "timezone": timezone,
    "protocol_rules": Rules.into_default()
    if protocols is None
    else Rules.from_dict(protocols),
    "msg_type_event_types": registry.msg_type_event_types(),
    "spill": spill,
    **({} if header is None else {"header_pattern": header}),
}
location = Url.from_string(str(source)).resolve(project_root)
filesystem, path = resolve(location)
info = filesystem.get_file_info(path)
if info.type == pyarrow.fs.FileType.NotFound:
    raise FileNotFoundError(location)
rows = (
    TextFiles.from_folder(location, pattern=pattern, recursive=recursive, **declared)
    if info.type == pyarrow.fs.FileType.Directory
    else TextFile.from_url(location, **declared)
)
field = rows.into_struct_field()
stage = Stage(
    "parse_messages",
    sources={"capture": str(location)},
    targets={"messages": target},
    window=(lower, upper),
)

In [ ]:
messages = IcebergDataset(
    field=field.with_name(target),
    catalog=catalog,
    properties=dict(catalog_properties),
    table_properties=dict(table_properties),
    branch=branch,
    commit_row_size=commit_row_size,
    sort_by=("unix", "hash"),
)

counts = {"read": 0}
technical_plugin_values = pyarrow.array(
    sorted({str(value).lower() for value in technical_plugins or ()}),
    pyarrow.string(),
)


def _batches():
    reader = rows.read_arrow_reader(
        batch_row_size=batch_row_size,
        batch_byte_size=batch_byte_size,
        max_row_byte_size=max_row_byte_size,
        include_regexes=include_regexes,
        exclude_regexes=exclude_regexes,
        include_msgtypes=include_msgtypes,
        exclude_msgtypes=exclude_msgtypes,
        start_unix=lower,
        end_unix=upper,
        duration_ns=duration_ns,
    )
    try:
        for batch in reader:
            if len(technical_plugin_values):
                excluded = pc.is_in(
                    pc.utf8_lower(batch.column("plugincode")),
                    value_set=technical_plugin_values,
                )
                batch = batch.filter(pc.invert(pc.fill_null(excluded, False)))
            if limit is not None and counts["read"] + batch.num_rows > limit:
                batch = batch.slice(0, max(0, limit - counts["read"]))
            if batch.num_rows:
                counts["read"] += batch.num_rows
                yield batch
            if limit is not None and counts["read"] >= limit:
                break
    finally:
        reader.close()


written = messages.append_arrow_reader(
    _batches(), field, merge_by=merge_by, commit_row_size=commit_row_size
)
result = stage.finished(read=counts["read"], written=written)
result